In [ ]:
!pip install implicit

import os
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import sparse
import implicit
from collections import defaultdict

from catboost import CatBoostRanker, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score

# =========================================================
# 1. КОНФИГ
# =========================================================
DATA_DIR = "/kaggle/input/datasets/tickflag/nto-ai-final/"
RANDOM_STATE = 1488
np.random.seed(RANDOM_STATE)

# Реальный инцидент
INCIDENT_START = pd.Timestamp("2025-10-01 00:00:00")
INCIDENT_END   = pd.Timestamp("2025-11-01 00:00:00")

# Псевдо-инцидент для локального обучения
PSEUDO_START = pd.Timestamp("2025-09-01 00:00:00")
PSEUDO_END   = pd.Timestamp("2025-10-01 00:00:00")

# Доля скрываемых пар
HIDE_FRAC = 0.25

# Candidate sizes
N_CANDIDATES_ALS = 150
N_CANDIDATES_BM25 = 80
N_CANDIDATES_POP = 80
N_CANDIDATES_I2I = 120
N_CANDIDATES_AUTHOR = 80
N_CANDIDATES_GENRE = 80
N_CANDIDATES_BOOK = 120

# observed-window generators
N_CANDIDATES_WINDOW_I2I = 80
N_CANDIDATES_WINDOW_COVISIT = 80
N_CANDIDATES_WINDOW_AUTHOR = 60

# exposure generator
N_CANDIDATES_EXPOSURE = 100

I2I_RECENT_K = 5
I2I_TOPN_NEIGH = 80
TRAIN_USERS_SAMPLE = 20000

CB_PARAMS = dict(
    iterations=2500,
    depth=8,
    learning_rate=0.05,
    loss_function="YetiRankPairwise",
    eval_metric="NDCG:top=20;type=Base",
    random_seed=RANDOM_STATE,
    task_type="GPU",
    verbose=100
)

tqdm.pandas()

# =========================================================
# 2. ЗАГРУЗКА
# =========================================================
print("1. Загружаю данные...")

interactions = pd.read_csv(os.path.join(DATA_DIR, "interactions.csv"), parse_dates=["event_ts"])
users = pd.read_csv(os.path.join(DATA_DIR, "users.csv"))
editions = pd.read_csv(os.path.join(DATA_DIR, "editions.csv"))
targets = pd.read_csv(os.path.join(DATA_DIR, "targets.csv"))
book_genres = pd.read_csv(os.path.join(DATA_DIR, "book_genres.csv"))
authors = pd.read_csv(os.path.join(DATA_DIR, "authors.csv"))

# только позитивные события
interactions = interactions[interactions["event_type"].isin([1, 2])].copy()

users["age"] = users["age"].fillna(users["age"].median())
editions["publication_year"] = editions["publication_year"].fillna(editions["publication_year"].median())
editions["age_restriction"] = editions["age_restriction"].fillna(0)

max_ts = interactions["event_ts"].max()

print(f"Максимальная дата: {max_ts}")
print(f"Реальный инцидент: [{INCIDENT_START} ; {INCIDENT_END})")
print(f"Псевдо-инцидент:   [{PSEUDO_START} ; {PSEUDO_END})")

edition_meta = editions[
    ["edition_id", "book_id", "author_id", "publication_year", "age_restriction", "language_id", "publisher_id"]
].copy()

edition_to_book = edition_meta.set_index("edition_id")["book_id"].to_dict()
edition_to_author = edition_meta.set_index("edition_id")["author_id"].to_dict()

# user/item mappings
user_cat = interactions["user_id"].astype("category")
item_cat = interactions["edition_id"].astype("category")
user_to_idx = dict(zip(user_cat, user_cat.cat.codes))
item_to_idx = dict(zip(item_cat, item_cat.cat.codes))
idx_to_item = {v: k for k, v in item_to_idx.items()}

# book mappings
all_books = pd.Series(interactions["edition_id"].map(edition_to_book).dropna().unique())
book_to_idx = {b: i for i, b in enumerate(all_books)}
idx_to_book = {i: b for b, i in book_to_idx.items()}

# =========================================================
# 3. МЕТРИКИ И УТИЛИТЫ
# =========================================================
def calc_ndcg_per_user(df_scored, score_col="pred", label_col="label", user_col="user_id", k=20):
    vals = []
    for _, g in df_scored.groupby(user_col):
        if g[label_col].sum() <= 0:
            continue
        y_true = g[label_col].values.reshape(1, -1)
        y_pred = g[score_col].values.reshape(1, -1)
        vals.append(ndcg_score(y_true, y_pred, k=k))
    return float(np.mean(vals)) if vals else 0.0


def calc_recall_at_k(cands_df, truth_pairs_df, k=100):
    truth_map = truth_pairs_df.groupby("user_id")["edition_id"].apply(set).to_dict()
    recalls = []

    for u, g in cands_df.groupby("user_id"):
        true_items = truth_map.get(u, None)
        if true_items is None or len(true_items) == 0:
            continue
        pred = g["edition_id"].head(k).tolist()
        recalls.append(len(set(pred) & true_items) / len(true_items))

    return float(np.mean(recalls)) if recalls else 0.0


def anti_join_pairs(left_df, right_df):
    tmp = left_df.merge(right_df.assign(_flag=1), on=["user_id", "edition_id"], how="left")
    return tmp[tmp["_flag"] != 1][["user_id", "edition_id"]]


# =========================================================
# 4. ПРАВИЛЬНАЯ ЛОКАЛЬНАЯ СИМУЛЯЦИЯ
# =========================================================
def sample_hidden_pairs(window_pairs_df, outside_pairs_df, hide_frac=0.25, seed=42):
    rng = np.random.default_rng(seed)

    eligible = anti_join_pairs(window_pairs_df, outside_pairs_df).drop_duplicates()
    hidden_rows = []

    for u, g in eligible.groupby("user_id"):
        items = g["edition_id"].tolist()
        n = len(items)
        if n == 0:
            continue

        if n == 1:
            hide_n = 1 if rng.random() < hide_frac else 0
        else:
            hide_n = max(1, int(round(hide_frac * n)))
            hide_n = min(hide_n, n)

        if hide_n <= 0:
            continue

        chosen = rng.choice(items, size=hide_n, replace=False)
        for eid in chosen:
            hidden_rows.append((u, eid))

    hidden_df = pd.DataFrame(hidden_rows, columns=["user_id", "edition_id"]).drop_duplicates()
    hidden_df["label"] = 1
    return hidden_df


def remove_hidden_events_from_window(window_events_df, hidden_pairs_df):
    tmp = window_events_df.merge(hidden_pairs_df.assign(_hide=1), on=["user_id", "edition_id"], how="left")
    return tmp[tmp["_hide"] != 1].drop(columns=["_hide"])


# =========================================================
# 5. RETRIEVAL MODELS
# =========================================================
def train_implicit_models(df_inter, ref_ts):
    df = df_inter.copy()
    df["days_ago"] = (ref_ts - df["event_ts"]).dt.days
    df = df[df["days_ago"] >= 0].copy()

    df["weight"] = np.where(df["event_type"] == 2, 3.0, 1.0) * np.exp(-0.02 * df["days_ago"])

    row = df["user_id"].map(user_to_idx).fillna(-1).astype(int)
    col = df["edition_id"].map(item_to_idx).fillna(-1).astype(int)
    mask = (row >= 0) & (col >= 0)

    row = row[mask].values
    col = col[mask].values
    val = (df.loc[mask, "weight"].values * 40).astype(np.float32)

    mat = sparse.csr_matrix(
        (val, (row, col)),
        shape=(len(user_to_idx), len(item_to_idx))
    ).tocsr()

    als = implicit.als.AlternatingLeastSquares(
        factors=128,
        regularization=0.05,
        iterations=20,
        random_state=RANDOM_STATE
    )
    als.fit(mat, show_progress=False)

    bm25 = implicit.nearest_neighbours.BM25Recommender(K=100)
    bm25.fit(mat, show_progress=False)

    return als, bm25, mat


def train_book_als(df_inter, ref_ts):
    df = df_inter.copy()
    df["book_id"] = df["edition_id"].map(edition_to_book)
    df = df[df["book_id"].notna()].copy()

    df["days_ago"] = (ref_ts - df["event_ts"]).dt.days
    df = df[df["days_ago"] >= 0].copy()

    df["weight"] = np.where(df["event_type"] == 2, 3.0, 1.0) * np.exp(-0.02 * df["days_ago"])

    agg = df.groupby(["user_id", "book_id"])["weight"].sum().reset_index()

    row = agg["user_id"].map(user_to_idx).fillna(-1).astype(int)
    col = agg["book_id"].map(book_to_idx).fillna(-1).astype(int)
    mask = (row >= 0) & (col >= 0)

    row = row[mask].values
    col = col[mask].values
    val = (agg.loc[mask, "weight"].values * 40).astype(np.float32)

    mat = sparse.csr_matrix(
        (val, (row, col)),
        shape=(len(user_to_idx), len(book_to_idx))
    ).tocsr()

    book_als = implicit.als.AlternatingLeastSquares(
        factors=96,
        regularization=0.05,
        iterations=15,
        random_state=RANDOM_STATE
    )
    book_als.fit(mat, show_progress=False)

    return book_als, mat


# =========================================================
# 6. CANDIDATE GENERATORS
# =========================================================
def get_cf_candidates(model, sparse_mat, user_ids, n=100, score_name="score"):
    valid_users = [u for u in user_ids if u in user_to_idx]
    if not valid_users:
        return pd.DataFrame(columns=["user_id", "edition_id", score_name])

    uidx = [user_to_idx[u] for u in valid_users]

    recs, scores = model.recommend(
        uidx,
        sparse_mat[uidx],
        N=n,
        filter_already_liked_items=True
    )

    recs = np.asarray(recs)
    scores = np.asarray(scores)

    if recs.ndim == 1:
        recs = recs.reshape(1, -1)
        scores = scores.reshape(1, -1)

    users_flat = np.repeat(valid_users, recs.shape[1])
    items_flat = recs.flatten()
    scores_flat = scores.flatten()

    n_items_total = len(idx_to_item)
    valid_mask = (
        (items_flat >= 0) &
        (items_flat < n_items_total)
    )

    dropped = int((~valid_mask).sum())
    if dropped > 0:
        print(f"[WARN] {score_name}: отброшено невалидных индексов = {dropped}")

    items_valid = [idx_to_item[int(i)] for i in items_flat[valid_mask]]

    return pd.DataFrame({
        "user_id": users_flat[valid_mask],
        "edition_id": items_valid,
        score_name: scores_flat[valid_mask]
    })


def build_i2i_dict(history_df, als_model, topn=80):
    item_ids = history_df["edition_id"].drop_duplicates().tolist()
    item_indices = [item_to_idx[i] for i in item_ids if i in item_to_idx]

    sim_dict = {}
    for idx in tqdm(item_indices, desc="Строю i2i"):
        try:
            ids, scores = als_model.similar_items(idx, N=topn + 1)
            out = []
            for j, s in zip(ids, scores):
                if j == idx:
                    continue
                out.append((idx_to_item[j], float(s)))
            sim_dict[idx_to_item[idx]] = out[:topn]
        except Exception:
            sim_dict[idx_to_item[idx]] = []

    return sim_dict


def get_i2i_candidates(user_ids, history_df, i2i_dict, n=100, recent_k=5, score_name="score_i2i"):
    hist = history_df.sort_values(["user_id", "event_ts"]).copy()
    user_items = hist.groupby("user_id")["edition_id"].apply(list).to_dict()
    seen_map = hist.groupby("user_id")["edition_id"].apply(set).to_dict()

    rows = []
    for u in tqdm(user_ids, desc="Генерирую i2i-кандидаты"):
        items = user_items.get(u, [])
        if not items:
            continue

        recent_items = items[-recent_k:]
        cand_scores = {}

        for pos, src in enumerate(reversed(recent_items)):
            w = 1.0 / (1.0 + 0.25 * pos)
            for cand, sim in i2i_dict.get(src, []):
                if cand in seen_map.get(u, set()):
                    continue
                cand_scores[cand] = cand_scores.get(cand, 0.0) + w * sim

        top_items = sorted(cand_scores.items(), key=lambda x: x[1], reverse=True)[:n]
        for eid, sc in top_items:
            rows.append((u, eid, float(sc)))

    if not rows:
        return pd.DataFrame(columns=["user_id", "edition_id", score_name])

    return pd.DataFrame(rows, columns=["user_id", "edition_id", score_name])


def get_pop_candidates(user_ids, pop_items, n=100, score_name="score_pop"):
    pop_items = list(pop_items)[:n]
    if not pop_items:
        return pd.DataFrame(columns=["user_id", "edition_id", score_name])

    score_map = {eid: (n - r) / n for r, eid in enumerate(pop_items)}
    rows = []
    for u in user_ids:
        for eid in pop_items:
            rows.append((u, eid, score_map[eid]))
    return pd.DataFrame(rows, columns=["user_id", "edition_id", score_name])


def build_author_top_items(history_df, recent_days=60, top_per_author=40):
    cutoff = history_df["event_ts"].max() - pd.Timedelta(days=recent_days)
    recent = history_df[history_df["event_ts"] >= cutoff].copy()
    if recent.empty:
        recent = history_df.copy()

    recent = recent.merge(edition_meta[["edition_id", "author_id"]], on="edition_id", how="left")
    agg = recent.groupby(["author_id", "edition_id"]).size().reset_index(name="cnt")
    agg = agg.sort_values(["author_id", "cnt"], ascending=[True, False])

    author_top = {}
    for a, g in agg.groupby("author_id"):
        author_top[a] = g["edition_id"].head(top_per_author).tolist()
    return author_top


def get_author_candidates(user_ids, history_df, author_top_items, n=80, score_name="score_author"):
    hist = history_df.merge(edition_meta[["edition_id", "author_id"]], on="edition_id", how="left")
    user_author_cnt = hist.groupby(["user_id", "author_id"]).size().reset_index(name="cnt")
    user_author_cnt = user_author_cnt.sort_values(["user_id", "cnt"], ascending=[True, False])

    seen_map = history_df.groupby("user_id")["edition_id"].apply(set).to_dict()
    user_ids_set = set(user_ids)
    rows = []

    for u, g in tqdm(user_author_cnt.groupby("user_id"), desc="Генерирую author-кандидаты"):
        if u not in user_ids_set:
            continue

        cand_scores = {}
        top_authors = g["author_id"].head(5).tolist()

        for rank_a, a in enumerate(top_authors):
            aw = 1.0 / (1 + rank_a)
            for rank_e, eid in enumerate(author_top_items.get(a, [])):
                if eid in seen_map.get(u, set()):
                    continue
                cand_scores[eid] = cand_scores.get(eid, 0.0) + aw * (1.0 / (1 + rank_e))

        top_items = sorted(cand_scores.items(), key=lambda x: x[1], reverse=True)[:n]
        for eid, sc in top_items:
            rows.append((u, eid, float(sc)))

    if not rows:
        return pd.DataFrame(columns=["user_id", "edition_id", score_name])

    return pd.DataFrame(rows, columns=["user_id", "edition_id", score_name])


def build_genre_top_items(history_df, recent_days=60, top_per_genre=50):
    cutoff = history_df["event_ts"].max() - pd.Timedelta(days=recent_days)
    recent = history_df[history_df["event_ts"] >= cutoff].copy()
    if recent.empty:
        recent = history_df.copy()

    recent = recent.merge(edition_meta[["edition_id", "book_id"]], on="edition_id", how="left")
    recent = recent.merge(book_genres, on="book_id", how="left")

    agg = recent.groupby(["genre_id", "edition_id"]).size().reset_index(name="cnt")
    agg = agg.sort_values(["genre_id", "cnt"], ascending=[True, False])

    genre_top = {}
    for gid, g in agg.groupby("genre_id"):
        genre_top[gid] = g["edition_id"].head(top_per_genre).tolist()
    return genre_top


def get_genre_candidates(user_ids, history_df, genre_top_items, n=80, score_name="score_genre_gen"):
    hist = history_df.merge(edition_meta[["edition_id", "book_id"]], on="edition_id", how="left")
    hist = hist.merge(book_genres, on="book_id", how="left")

    user_genre_cnt = hist.groupby(["user_id", "genre_id"]).size().reset_index(name="cnt")
    user_genre_cnt = user_genre_cnt.sort_values(["user_id", "cnt"], ascending=[True, False])

    seen_map = history_df.groupby("user_id")["edition_id"].apply(set).to_dict()
    user_ids_set = set(user_ids)
    rows = []

    for u, g in tqdm(user_genre_cnt.groupby("user_id"), desc="Генерирую genre-кандидаты"):
        if u not in user_ids_set:
            continue

        cand_scores = {}
        top_genres = g["genre_id"].head(5).tolist()

        for rank_g, gid in enumerate(top_genres):
            gw = 1.0 / (1 + rank_g)
            for rank_e, eid in enumerate(genre_top_items.get(gid, [])):
                if eid in seen_map.get(u, set()):
                    continue
                cand_scores[eid] = cand_scores.get(eid, 0.0) + gw * (1.0 / (1 + rank_e))

        top_items = sorted(cand_scores.items(), key=lambda x: x[1], reverse=True)[:n]
        for eid, sc in top_items:
            rows.append((u, eid, float(sc)))

    if not rows:
        return pd.DataFrame(columns=["user_id", "edition_id", score_name])

    df = pd.DataFrame(rows, columns=["user_id", "edition_id", score_name])
    return df.groupby(["user_id", "edition_id"], as_index=False)[score_name].sum()


def build_book_top_editions(history_df, recent_days=120, top_per_book=3):
    hist = history_df.copy()
    cutoff = hist["event_ts"].max() - pd.Timedelta(days=recent_days)
    recent = hist[hist["event_ts"] >= cutoff].copy()
    if recent.empty:
        recent = hist.copy()

    recent["book_id"] = recent["edition_id"].map(edition_to_book)
    agg = recent.groupby(["book_id", "edition_id"]).size().reset_index(name="cnt")
    agg = agg.sort_values(["book_id", "cnt"], ascending=[True, False])

    book_top = {}
    for b, g in agg.groupby("book_id"):
        book_top[b] = g["edition_id"].head(top_per_book).tolist()
    return book_top


def get_book_candidates(book_model, sparse_book, user_ids, book_top_editions, n_books=80, editions_per_book=2, score_name="score_book_gen"):
    valid_users = [u for u in user_ids if u in user_to_idx]
    if not valid_users:
        return pd.DataFrame(columns=["user_id", "edition_id", score_name])

    uidx = [user_to_idx[u] for u in valid_users]
    recs, scores = book_model.recommend(
        uidx,
        sparse_book[uidx],
        N=n_books,
        filter_already_liked_items=True
    )

    recs = np.asarray(recs)
    scores = np.asarray(scores)
    if recs.ndim == 1:
        recs = recs.reshape(1, -1)
        scores = scores.reshape(1, -1)

    rows = []
    n_books_total = len(idx_to_book)

    for i, u in enumerate(valid_users):
        for j in range(recs.shape[1]):
            book_idx = int(recs[i, j])
            if book_idx < 0 or book_idx >= n_books_total:
                continue

            book_id = idx_to_book[book_idx]
            book_score = float(scores[i, j])

            top_editions = book_top_editions.get(book_id, [])[:editions_per_book]
            for r, eid in enumerate(top_editions):
                rows.append((u, eid, book_score / (1 + r)))

    if not rows:
        return pd.DataFrame(columns=["user_id", "edition_id", score_name])

    df = pd.DataFrame(rows, columns=["user_id", "edition_id", score_name])
    return df.groupby(["user_id", "edition_id"], as_index=False)[score_name].max()


def merge_sources(*dfs):
    out = None
    for df in dfs:
        if df is None or len(df) == 0:
            continue
        if out is None:
            out = df.copy()
        else:
            out = out.merge(df, on=["user_id", "edition_id"], how="outer")
    if out is None:
        out = pd.DataFrame(columns=["user_id", "edition_id"])
    return out.fillna(0)


# =========================================================
# 6.5. ЭВРИСТИКА КАК ФИЧА
# =========================================================
def build_heuristic_artifacts(history_df, ref_ts, user_recent_k=5, covisit_window=4, pop_topk=300):
    print("Строю артефакты эвристики для фичей...")

    hist = history_df[history_df["event_ts"] < ref_ts].copy()
    hist = hist.sort_values(["user_id", "event_ts"])

    last_user_items = hist.groupby("user_id")["edition_id"].apply(lambda x: list(x)[-user_recent_k:]).to_dict()

    co_visitation_matrix = defaultdict(lambda: defaultdict(int))
    for _, group in tqdm(hist.groupby("user_id"), desc="Эвристика: co-vis matrix"):
        items = group["edition_id"].tolist()
        for i in range(len(items)):
            for j in range(i + 1, min(i + 1 + covisit_window, len(items))):
                a = items[i]
                b = items[j]
                co_visitation_matrix[a][b] += 1
                co_visitation_matrix[b][a] += 1

    recent_hist = hist[hist["event_ts"] >= (ref_ts - pd.Timedelta(days=30))]
    recent_top = recent_hist["edition_id"].value_counts().head(pop_topk).index.tolist()
    pop_rank_map = {eid: rank + 1 for rank, eid in enumerate(recent_top)}

    return {
        "last_user_items": last_user_items,
        "co_visitation_matrix": co_visitation_matrix,
        "pop_rank_map": pop_rank_map
    }


def add_heuristic_features(cands_df, heur_artifacts):
    print("Добавляю эвристические фичи...")

    last_user_items = heur_artifacts["last_user_items"]
    co_visitation_matrix = heur_artifacts["co_visitation_matrix"]
    pop_rank_map = heur_artifacts["pop_rank_map"]

    parts = []

    for u_id, group in tqdm(cands_df.groupby("user_id"), desc="Эвристика: фичи по кандидатам"):
        cand_items = group["edition_id"].tolist()
        recent_items = list(reversed(last_user_items.get(u_id, [])))

        covisit_scores = []
        for cand in cand_items:
            score = 0.0
            for pos, src in enumerate(recent_items):
                w = 1.0 / (1.0 + 0.25 * pos)
                score += w * co_visitation_matrix.get(src, {}).get(cand, 0)
            covisit_scores.append(score)

        tmp = pd.DataFrame({
            "user_id": u_id,
            "edition_id": cand_items,
            "heur_covisit_score": covisit_scores
        })

        tmp["heur_pop_inv_rank"] = tmp["edition_id"].map(
            lambda x: 1.0 / (1.0 + pop_rank_map[x]) if x in pop_rank_map else 0.0
        )

        tmp["heur_score_total"] = tmp["heur_covisit_score"] + 5.0 * tmp["heur_pop_inv_rank"]

        tmp = tmp.sort_values(
            ["heur_score_total", "heur_covisit_score", "heur_pop_inv_rank"],
            ascending=[False, False, False]
        ).reset_index(drop=True)

        tmp["heur_rank"] = np.arange(1, len(tmp) + 1)
        tmp["heur_rank_inv"] = 1.0 / (1.0 + tmp["heur_rank"])
        tmp["heur_top20"] = (tmp["heur_rank"] <= 20).astype(int)
        tmp["heur_top50"] = (tmp["heur_rank"] <= 50).astype(int)

        parts.append(
            tmp[[
                "user_id", "edition_id",
                "heur_covisit_score",
                "heur_pop_inv_rank",
                "heur_rank_inv",
                "heur_top20",
                "heur_top50"
            ]]
        )

    heur_df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
        columns=[
            "user_id", "edition_id",
            "heur_covisit_score",
            "heur_pop_inv_rank",
            "heur_rank_inv",
            "heur_top20",
            "heur_top50"
        ]
    )

    out = cands_df.merge(heur_df, on=["user_id", "edition_id"], how="left")
    for col in ["heur_covisit_score", "heur_pop_inv_rank", "heur_rank_inv", "heur_top20", "heur_top50"]:
        out[col] = out[col].fillna(0)

    return out


# =========================================================
# 6.6. OBSERVED-WINDOW ARTIFACTS
# =========================================================
def build_window_artifacts(window_obs_df):
    print("Строю артефакты observed-window...")

    if window_obs_df is None or len(window_obs_df) == 0:
        return {
            "window_items_map": {},
            "window_author_cnt_df": pd.DataFrame(columns=["user_id", "author_id", "obs_window_same_author_cnt"]),
            "window_book_cnt_df": pd.DataFrame(columns=["user_id", "book_id", "obs_window_same_book_cnt"]),
            "window_user_genre_cnt_df": pd.DataFrame(columns=["user_id", "genre_id", "obs_window_genre_inter_cnt"]),
            "window_user_cnt_df": pd.DataFrame(columns=["user_id", "obs_window_item_cnt"]),
            "window_pop_rank_map": {},
            "window_item_pop_df": pd.DataFrame(columns=["edition_id", "window_item_pop"]),
            "window_author_map": {},
            "window_book_map": {},
            "window_genre_map": {},
        }

    w = window_obs_df.sort_values(["user_id", "event_ts"]).copy()
    window_items_map = w.groupby("user_id")["edition_id"].apply(list).to_dict()

    w_ext = w.merge(edition_meta[["edition_id", "author_id", "book_id"]], on="edition_id", how="left")

    window_author_cnt_df = (
        w_ext.groupby(["user_id", "author_id"])
        .size()
        .reset_index(name="obs_window_same_author_cnt")
    )

    window_book_cnt_df = (
        w_ext.groupby(["user_id", "book_id"])
        .size()
        .reset_index(name="obs_window_same_book_cnt")
    )

    w_gen = w_ext.merge(book_genres, on="book_id", how="left")
    window_user_genre_cnt_df = (
        w_gen.groupby(["user_id", "genre_id"])
        .size()
        .reset_index(name="obs_window_genre_inter_cnt")
    )

    window_user_cnt_df = (
        w.groupby("user_id")
        .size()
        .reset_index(name="obs_window_item_cnt")
    )

    window_item_pop_df = (
        w.groupby("edition_id")
        .size()
        .reset_index(name="window_item_pop")
    )

    top_window_items = window_item_pop_df.sort_values("window_item_pop", ascending=False)["edition_id"].tolist()
    window_pop_rank_map = {eid: rank + 1 for rank, eid in enumerate(top_window_items)}

    window_author_map = (
        window_author_cnt_df.sort_values(["user_id", "obs_window_same_author_cnt"], ascending=[True, False])
        .groupby("user_id")
        .apply(lambda d: list(zip(d["author_id"], d["obs_window_same_author_cnt"])))
        .to_dict()
    )

    window_book_map = (
        window_book_cnt_df.sort_values(["user_id", "obs_window_same_book_cnt"], ascending=[True, False])
        .groupby("user_id")
        .apply(lambda d: list(zip(d["book_id"], d["obs_window_same_book_cnt"])))
        .to_dict()
    )

    window_genre_map = (
        window_user_genre_cnt_df.sort_values(["user_id", "obs_window_genre_inter_cnt"], ascending=[True, False])
        .groupby("user_id")
        .apply(lambda d: list(zip(d["genre_id"], d["obs_window_genre_inter_cnt"])))
        .to_dict()
    )

    return {
        "window_items_map": window_items_map,
        "window_author_cnt_df": window_author_cnt_df,
        "window_book_cnt_df": window_book_cnt_df,
        "window_user_genre_cnt_df": window_user_genre_cnt_df,
        "window_user_cnt_df": window_user_cnt_df,
        "window_pop_rank_map": window_pop_rank_map,
        "window_item_pop_df": window_item_pop_df,
        "window_author_map": window_author_map,
        "window_book_map": window_book_map,
        "window_genre_map": window_genre_map,
    }


# =========================================================
# 6.7. OBSERVED-WINDOW GENERATORS
# =========================================================
def get_window_i2i_candidates(user_ids, history_df, window_artifacts, i2i_dict, n=80, recent_k=10, score_name="score_window_i2i"):
    print("Генерирую observed-window i2i кандидаты...")

    window_items_map = window_artifacts["window_items_map"]
    seen_map = history_df.groupby("user_id")["edition_id"].apply(set).to_dict()

    rows = []
    for u in tqdm(user_ids, desc="window_i2i"):
        items = window_items_map.get(u, [])
        if not items:
            continue

        cand_scores = {}
        for pos, src in enumerate(reversed(items[-recent_k:])):
            w = 1.0 / (1.0 + 0.25 * pos)
            for cand, sim in i2i_dict.get(src, []):
                if cand in seen_map.get(u, set()):
                    continue
                cand_scores[cand] = cand_scores.get(cand, 0.0) + w * sim

        top_items = sorted(cand_scores.items(), key=lambda x: x[1], reverse=True)[:n]
        for eid, sc in top_items:
            rows.append((u, eid, float(sc)))

    if not rows:
        return pd.DataFrame(columns=["user_id", "edition_id", score_name])

    return pd.DataFrame(rows, columns=["user_id", "edition_id", score_name])


def get_window_covisit_candidates(user_ids, history_df, window_artifacts, co_visitation_matrix, n=80, recent_k=10, score_name="score_window_covisit"):
    print("Генерирую observed-window covisit кандидаты...")

    window_items_map = window_artifacts["window_items_map"]
    seen_map = history_df.groupby("user_id")["edition_id"].apply(set).to_dict()

    rows = []
    for u in tqdm(user_ids, desc="window_covisit"):
        items = window_items_map.get(u, [])
        if not items:
            continue

        cand_scores = {}
        for pos, src in enumerate(reversed(items[-recent_k:])):
            w = 1.0 / (1.0 + 0.25 * pos)
            neighbors = co_visitation_matrix.get(src, {})
            for cand, cnt in neighbors.items():
                if cand in seen_map.get(u, set()):
                    continue
                cand_scores[cand] = cand_scores.get(cand, 0.0) + w * cnt

        top_items = sorted(cand_scores.items(), key=lambda x: x[1], reverse=True)[:n]
        for eid, sc in top_items:
            rows.append((u, eid, float(sc)))

    if not rows:
        return pd.DataFrame(columns=["user_id", "edition_id", score_name])

    return pd.DataFrame(rows, columns=["user_id", "edition_id", score_name])


def get_window_author_candidates(user_ids, history_df, window_artifacts, author_top_items, n=60, score_name="score_window_author"):
    print("Генерирую observed-window author кандидаты...")

    seen_map = history_df.groupby("user_id")["edition_id"].apply(set).to_dict()
    user_ids_set = set(user_ids)
    window_author_map = window_artifacts["window_author_map"]

    rows = []
    for u, author_list in tqdm(window_author_map.items(), desc="window_author"):
        if u not in user_ids_set:
            continue

        cand_scores = {}
        top_authors = author_list[:5]

        for rank_a, (a, cnt) in enumerate(top_authors):
            aw = cnt / (1 + rank_a)
            for rank_e, eid in enumerate(author_top_items.get(a, [])[:30]):
                if eid in seen_map.get(u, set()):
                    continue
                cand_scores[eid] = cand_scores.get(eid, 0.0) + aw / (1 + rank_e)

        top_items = sorted(cand_scores.items(), key=lambda x: x[1], reverse=True)[:n]
        for eid, sc in top_items:
            rows.append((u, eid, float(sc)))

    if not rows:
        return pd.DataFrame(columns=["user_id", "edition_id", score_name])

    return pd.DataFrame(rows, columns=["user_id", "edition_id", score_name])


# =========================================================
# 6.8. EXPOSURE GENERATOR
# =========================================================
def get_exposure_candidates(
    user_ids,
    history_df,
    window_artifacts,
    author_top_items,
    genre_top_items,
    book_top_editions,
    n=100,
    score_name="score_exposure"
):
    print("Генерирую exposure кандидаты...")

    seen_map = history_df.groupby("user_id")["edition_id"].apply(set).to_dict()
    user_ids_set = set(user_ids)

    window_author_map = window_artifacts["window_author_map"]
    window_book_map = window_artifacts["window_book_map"]
    window_genre_map = window_artifacts["window_genre_map"]
    window_pop_rank_map = window_artifacts["window_pop_rank_map"]

    top_window_pop_items = list(window_pop_rank_map.keys())[:150]

    rows = []
    for u in tqdm(user_ids, desc="exposure"):
        if u not in user_ids_set:
            continue

        cand_scores = {}

        # same book
        for rank_b, (book_id, cnt) in enumerate(window_book_map.get(u, [])[:8]):
            bw = 3.0 * cnt / (1 + rank_b)
            for rank_e, eid in enumerate(book_top_editions.get(book_id, [])[:3]):
                if eid in seen_map.get(u, set()):
                    continue
                cand_scores[eid] = cand_scores.get(eid, 0.0) + bw / (1 + rank_e)

        # same author
        for rank_a, (author_id, cnt) in enumerate(window_author_map.get(u, [])[:5]):
            aw = 2.0 * cnt / (1 + rank_a)
            for rank_e, eid in enumerate(author_top_items.get(author_id, [])[:25]):
                if eid in seen_map.get(u, set()):
                    continue
                cand_scores[eid] = cand_scores.get(eid, 0.0) + aw / (1 + rank_e)

        # same genre
        for rank_g, (genre_id, cnt) in enumerate(window_genre_map.get(u, [])[:6]):
            gw = 1.2 * cnt / (1 + rank_g)
            for rank_e, eid in enumerate(genre_top_items.get(genre_id, [])[:25]):
                if eid in seen_map.get(u, set()):
                    continue
                cand_scores[eid] = cand_scores.get(eid, 0.0) + gw / (1 + rank_e)

        # popularity inside observed window
        for rank_p, eid in enumerate(top_window_pop_items[:80]):
            if eid in seen_map.get(u, set()):
                continue
            cand_scores[eid] = cand_scores.get(eid, 0.0) + 0.35 / (1 + rank_p)

        # extra pop boost
        for eid in list(cand_scores.keys()):
            if eid in window_pop_rank_map:
                cand_scores[eid] += 1.5 / (1 + window_pop_rank_map[eid])

        top_items = sorted(cand_scores.items(), key=lambda x: x[1], reverse=True)[:n]
        for eid, sc in top_items:
            rows.append((u, eid, float(sc)))

    if not rows:
        return pd.DataFrame(columns=["user_id", "edition_id", score_name])

    return pd.DataFrame(rows, columns=["user_id", "edition_id", score_name])


# =========================================================
# 7. ФИЧИ
# =========================================================
def extract_features(cands_df, history_df, ref_ts, i2i_dict, window_artifacts, heur_artifacts):
    hist = history_df.copy()
    hist["days_ago"] = (ref_ts - hist["event_ts"]).dt.days
    hist = hist[hist["days_ago"] >= 0].copy()

    co_visitation_matrix = heur_artifacts["co_visitation_matrix"]
    window_items_map = window_artifacts["window_items_map"]
    window_author_cnt_df = window_artifacts["window_author_cnt_df"]
    window_book_cnt_df = window_artifacts["window_book_cnt_df"]
    window_user_genre_cnt_df = window_artifacts["window_user_genre_cnt_df"]
    window_user_cnt_df = window_artifacts["window_user_cnt_df"]
    window_item_pop_df = window_artifacts["window_item_pop_df"]
    window_pop_rank_map = window_artifacts["window_pop_rank_map"]

    user_act_all = hist.groupby("user_id").size().reset_index(name="user_act_all")
    user_act_30d = hist[hist["days_ago"] <= 30].groupby("user_id").size().reset_index(name="user_act_30d")

    item_pop_all = hist.groupby("edition_id").size().reset_index(name="item_pop_all")
    item_pop_30d = hist[hist["days_ago"] <= 30].groupby("edition_id").size().reset_index(name="item_pop_30d")

    hist_book = hist.copy()
    hist_book["book_id"] = hist_book["edition_id"].map(edition_to_book)
    book_pop_all = hist_book.groupby("book_id").size().reset_index(name="book_pop_all")
    book_pop_30d = hist_book[hist_book["days_ago"] <= 30].groupby("book_id").size().reset_index(name="book_pop_30d")

    inter_ext = hist.merge(edition_meta[["edition_id", "author_id", "book_id"]], on="edition_id", how="left")
    user_author_cnt = inter_ext.groupby(["user_id", "author_id"]).size().reset_index(name="author_inter_count")

    inter_gen = inter_ext.merge(book_genres, on="book_id", how="left")
    user_genre_cnt = inter_gen.groupby(["user_id", "genre_id"]).size().reset_index(name="genre_inter_count")

    last_user_event = hist.groupby("user_id")["event_ts"].max().reset_index(name="last_user_event_ts")
    last_author_event = inter_ext.groupby(["user_id", "author_id"])["event_ts"].max().reset_index(name="last_author_event_ts")
    last_book_event = inter_ext.groupby(["user_id", "book_id"])["event_ts"].max().reset_index(name="last_book_event_ts")

    hist_sorted = hist.sort_values(["user_id", "event_ts"])
    recent_items_map = hist_sorted.groupby("user_id")["edition_id"].apply(list).to_dict()

    res = cands_df.merge(edition_meta, on="edition_id", how="left")
    res = res.merge(users[["user_id", "age"]], on="user_id", how="left")

    res = res.merge(user_act_all, on="user_id", how="left").fillna({"user_act_all": 0})
    res = res.merge(user_act_30d, on="user_id", how="left").fillna({"user_act_30d": 0})

    res = res.merge(item_pop_all, on="edition_id", how="left").fillna({"item_pop_all": 0})
    res = res.merge(item_pop_30d, on="edition_id", how="left").fillna({"item_pop_30d": 0})
    res = res.merge(book_pop_all, on="book_id", how="left").fillna({"book_pop_all": 0})
    res = res.merge(book_pop_30d, on="book_id", how="left").fillna({"book_pop_30d": 0})

    res = res.merge(user_author_cnt, on=["user_id", "author_id"], how="left").fillna({"author_inter_count": 0})
    res["same_author_seen_before"] = (res["author_inter_count"] > 0).astype(int)

    cand_gen = res[["user_id", "edition_id", "book_id"]].merge(book_genres, on="book_id", how="left")
    cand_gen = cand_gen.merge(user_genre_cnt, on=["user_id", "genre_id"], how="left")

    cand_gen["genre_inter_count"] = cand_gen["genre_inter_count"].fillna(0)
    cand_gen["genre_seen_bin"] = (cand_gen["genre_inter_count"] > 0).astype(int)

    genre_agg = (
        cand_gen
        .groupby(["user_id", "edition_id"])
        .agg(
            genre_score=("genre_inter_count", "sum"),
            matched_genres_cnt=("genre_seen_bin", "sum"),
            candidate_genres_cnt=("genre_id", "nunique")
        )
        .reset_index()
    )

    genre_agg["candidate_genres_cnt"] = genre_agg["candidate_genres_cnt"].replace(0, 1)
    genre_agg["genre_match_ratio"] = genre_agg["matched_genres_cnt"] / genre_agg["candidate_genres_cnt"]

    res = res.merge(
        genre_agg[["user_id", "edition_id", "genre_score", "genre_match_ratio"]],
        on=["user_id", "edition_id"],
        how="left"
    )

    res["genre_score"] = res["genre_score"].fillna(0)
    res["genre_match_ratio"] = res["genre_match_ratio"].fillna(0)

    user_book_seen = hist_book.groupby(["user_id", "book_id"]).size().reset_index(name="user_book_seen")
    res = res.merge(user_book_seen, on=["user_id", "book_id"], how="left")
    res["same_book_seen_before"] = (res["user_book_seen"].fillna(0) > 0).astype(int)
    res.drop(columns=["user_book_seen"], inplace=True)

    # observed-window counts
    res = res.merge(window_author_cnt_df, on=["user_id", "author_id"], how="left")
    res["obs_window_same_author_cnt"] = res["obs_window_same_author_cnt"].fillna(0)

    res = res.merge(window_book_cnt_df, on=["user_id", "book_id"], how="left")
    res["obs_window_same_book_cnt"] = res["obs_window_same_book_cnt"].fillna(0)

    cand_window_gen = res[["user_id", "edition_id", "book_id"]].merge(book_genres, on="book_id", how="left")
    cand_window_gen = cand_window_gen.merge(window_user_genre_cnt_df, on=["user_id", "genre_id"], how="left")
    obs_window_genre_agg = (
        cand_window_gen
        .groupby(["user_id", "edition_id"])["obs_window_genre_inter_cnt"]
        .sum()
        .reset_index(name="obs_window_same_genre_cnt")
    )
    res = res.merge(obs_window_genre_agg, on=["user_id", "edition_id"], how="left")
    res["obs_window_same_genre_cnt"] = res["obs_window_same_genre_cnt"].fillna(0)

    res = res.merge(window_user_cnt_df, on="user_id", how="left")
    res["obs_window_item_cnt"] = res["obs_window_item_cnt"].fillna(0)

    res = res.merge(window_item_pop_df, on="edition_id", how="left")
    res["window_item_pop"] = res["window_item_pop"].fillna(0)
    res["window_pop_rank_inv"] = res["edition_id"].map(
        lambda x: 1.0 / (1.0 + window_pop_rank_map[x]) if x in window_pop_rank_map else 0.0
    )

    # exposure-style features
    res["exposure_overlap_score"] = (
        3.0 * res["obs_window_same_book_cnt"] +
        2.0 * res["obs_window_same_author_cnt"] +
        1.0 * res["obs_window_same_genre_cnt"] +
        8.0 * res["window_pop_rank_inv"]
    )
    res["obs_window_activity_ratio"] = res["obs_window_item_cnt"] / (res["user_act_all"] + 1.0)

    res = res.merge(last_user_event, on="user_id", how="left")
    res = res.merge(last_author_event, on=["user_id", "author_id"], how="left")
    res = res.merge(last_book_event, on=["user_id", "book_id"], how="left")

    res["days_since_user_last_event"] = (ref_ts - res["last_user_event_ts"]).dt.days.fillna(9999)
    res["days_since_author_last_event"] = (ref_ts - res["last_author_event_ts"]).dt.days.fillna(9999)
    res["days_since_book_last_event"] = (ref_ts - res["last_book_event_ts"]).dt.days.fillna(9999)
    res.drop(columns=["last_user_event_ts", "last_author_event_ts", "last_book_event_ts"], inplace=True)

    def sim_stats_for_row(row):
        u = row["user_id"]
        cand = row["edition_id"]

        # общая история
        recent = recent_items_map.get(u, [])
        sims_last3 = []
        sims_last5_weighted = []
        recent_rev = list(reversed(recent[-5:]))

        for pos, src in enumerate(recent_rev):
            sim = dict(i2i_dict.get(src, [])).get(cand, 0.0)
            weight = 1.0 / (1.0 + 0.25 * pos)
            sims_last5_weighted.append(sim * weight)
            if pos < 3:
                sims_last3.append(sim)

        i2i_max_last3 = max(sims_last3) if sims_last3 else 0.0
        i2i_mean_last3 = float(np.mean(sims_last3)) if sims_last3 else 0.0
        i2i_sum_last5 = float(np.sum(sims_last5_weighted)) if sims_last5_weighted else 0.0

        # observed-window
        window_items = window_items_map.get(u, [])
        obs_window_i2i_vals = []
        obs_window_covisit_vals = []

        for pos, src in enumerate(reversed(window_items[-10:])):
            weight = 1.0 / (1.0 + 0.25 * pos)
            obs_window_i2i_vals.append(dict(i2i_dict.get(src, [])).get(cand, 0.0))
            obs_window_covisit_vals.append(weight * co_visitation_matrix.get(src, {}).get(cand, 0.0))

        obs_window_i2i_max = max(obs_window_i2i_vals) if obs_window_i2i_vals else 0.0
        obs_window_covisit_sum = float(np.sum(obs_window_covisit_vals)) if obs_window_covisit_vals else 0.0

        return pd.Series([
            i2i_max_last3,
            i2i_mean_last3,
            i2i_sum_last5,
            obs_window_i2i_max,
            obs_window_covisit_sum
        ])

    res[[
        "i2i_max_last3",
        "i2i_mean_last3",
        "i2i_sum_last5",
        "obs_window_i2i_max",
        "obs_window_covisit_sum"
    ]] = res.progress_apply(sim_stats_for_row, axis=1)

    res["activity_drop"] = res["user_act_30d"] / (res["user_act_all"] + 1.0)
    res["age_diff"] = res["age"] - res["age_restriction"]

    source_cols = [
        "score_als", "score_bm25", "score_pop", "score_i2i",
        "score_author", "score_genre_gen", "score_book_gen",
        "score_window_i2i", "score_window_covisit", "score_window_author",
        "score_exposure"
    ]
    for col in source_cols:
        if col not in res.columns:
            res[col] = 0.0

    res["n_generators_hit"] = (res[source_cols] > 0).sum(axis=1)
    return res


def add_propensity_weights(df):
    """
    Приближённый propensity / PU-weighting:
    - считаем propensity_score из exposure/window сигналов
    - positives: усиливаем inverse-propensity мягко
    - unlabeled: ослабляем там, где пара слишком похожа на hidden-positive
    """
    out = df.copy()

    max_window_pop = max(float(out["window_item_pop"].max()), 1.0)

    raw_prop = (
        2.5 * out["window_pop_rank_inv"].fillna(0).astype(float) +
        1.5 * (np.log1p(out["window_item_pop"].fillna(0).astype(float)) / np.log1p(max_window_pop)) +
        1.8 * np.log1p(out["obs_window_same_book_cnt"].fillna(0).astype(float)) +
        1.2 * np.log1p(out["obs_window_same_author_cnt"].fillna(0).astype(float)) +
        0.8 * np.log1p(out["obs_window_same_genre_cnt"].fillna(0).astype(float)) +
        0.8 * out["score_window_covisit"].fillna(0).astype(float) +
        0.8 * out["score_window_i2i"].fillna(0).astype(float) +
        0.6 * out["score_window_author"].fillna(0).astype(float)
    )

    rank_pct = raw_prop.rank(method="average", pct=True)
    out["propensity_score"] = (0.05 + 0.90 * rank_pct).astype(np.float32)

    pos_w = np.clip(1.0 / np.sqrt(out["propensity_score"]), 1.0, 3.0)
    unl_w = np.clip(1.0 - 0.70 * out["propensity_score"], 0.25, 1.0)

    out["sample_weight"] = np.where(
        out["label"].astype(int) == 1,
        pos_w,
        unl_w
    ).astype(np.float32)

    return out


def build_dataset(
    user_ids,
    history_df,
    ref_ts,
    label_pairs_df,
    als_model,
    bm25_model,
    sparse_mat,
    book_model,
    sparse_book,
    i2i_dict,
    author_top_items,
    genre_top_items,
    book_top_editions,
    pop_items,
    heur_artifacts,
    window_artifacts
):
    c_als = get_cf_candidates(als_model, sparse_mat, user_ids, n=N_CANDIDATES_ALS, score_name="score_als")
    c_bm25 = get_cf_candidates(bm25_model, sparse_mat, user_ids, n=N_CANDIDATES_BM25, score_name="score_bm25")
    c_pop = get_pop_candidates(user_ids, pop_items, n=N_CANDIDATES_POP, score_name="score_pop")
    c_i2i = get_i2i_candidates(user_ids, history_df, i2i_dict, n=N_CANDIDATES_I2I, recent_k=I2I_RECENT_K, score_name="score_i2i")
    c_author = get_author_candidates(user_ids, history_df, author_top_items, n=N_CANDIDATES_AUTHOR, score_name="score_author")
    c_genre = get_genre_candidates(user_ids, history_df, genre_top_items, n=N_CANDIDATES_GENRE, score_name="score_genre_gen")
    c_book = get_book_candidates(book_model, sparse_book, user_ids, book_top_editions, n_books=80, editions_per_book=2, score_name="score_book_gen")

    # observed-window generators
    c_window_i2i = get_window_i2i_candidates(
        user_ids, history_df, window_artifacts, i2i_dict,
        n=N_CANDIDATES_WINDOW_I2I, score_name="score_window_i2i"
    )
    c_window_covisit = get_window_covisit_candidates(
        user_ids, history_df, window_artifacts, heur_artifacts["co_visitation_matrix"],
        n=N_CANDIDATES_WINDOW_COVISIT, score_name="score_window_covisit"
    )
    c_window_author = get_window_author_candidates(
        user_ids, history_df, window_artifacts, author_top_items,
        n=N_CANDIDATES_WINDOW_AUTHOR, score_name="score_window_author"
    )

    # exposure generator
    c_exposure = get_exposure_candidates(
        user_ids=user_ids,
        history_df=history_df,
        window_artifacts=window_artifacts,
        author_top_items=author_top_items,
        genre_top_items=genre_top_items,
        book_top_editions=book_top_editions,
        n=N_CANDIDATES_EXPOSURE,
        score_name="score_exposure"
    )

    cands = merge_sources(
        c_als, c_bm25, c_pop, c_i2i, c_author, c_genre, c_book,
        c_window_i2i, c_window_covisit, c_window_author, c_exposure
    )

    cands = cands.merge(label_pairs_df, on=["user_id", "edition_id"], how="left")
    cands["label"] = pd.to_numeric(cands["label"], errors="coerce").fillna(0).astype(np.int8)

    cands = add_heuristic_features(cands, heur_artifacts)

    cands = extract_features(
        cands_df=cands,
        history_df=history_df,
        ref_ts=ref_ts,
        i2i_dict=i2i_dict,
        window_artifacts=window_artifacts,
        heur_artifacts=heur_artifacts
    )

    cands = add_propensity_weights(cands)
    cands = cands.sort_values("user_id").reset_index(drop=True)

    return (
        cands,
        c_als, c_bm25, c_pop, c_i2i, c_author, c_genre, c_book,
        c_window_i2i, c_window_covisit, c_window_author, c_exposure
    )


# =========================================================
# 8. ЛОКАЛЬНАЯ ПРАВИЛЬНАЯ СИМУЛЯЦИЯ
# =========================================================
print("\n2. Строю правильную локальную симуляцию...")

pre_pseudo = interactions[interactions["event_ts"] < PSEUDO_START].copy()
pseudo_window = interactions[(interactions["event_ts"] >= PSEUDO_START) & (interactions["event_ts"] < PSEUDO_END)].copy()
post_pseudo = interactions[interactions["event_ts"] >= PSEUDO_END].copy()

pseudo_pairs = pseudo_window[["user_id", "edition_id"]].drop_duplicates()
outside_pairs = pd.concat([
    pre_pseudo[["user_id", "edition_id"]].drop_duplicates(),
    post_pseudo[["user_id", "edition_id"]].drop_duplicates()
], ignore_index=True).drop_duplicates()

hidden_pairs = sample_hidden_pairs(pseudo_pairs, outside_pairs, hide_frac=HIDE_FRAC, seed=RANDOM_STATE)
print(f"Уникальных позитивных пар в псевдо-окне: {len(pseudo_pairs):,}")
print(f"Скрыто позитивных пар для обучения: {len(hidden_pairs):,}")

pseudo_observed = remove_hidden_events_from_window(pseudo_window, hidden_pairs[["user_id", "edition_id"]])
observed_train_log = pd.concat([pre_pseudo, pseudo_observed, post_pseudo], ignore_index=True)

print(f"Наблюдаемый лог для обучения: {len(observed_train_log):,} событий")

affected_users = hidden_pairs["user_id"].drop_duplicates().tolist()
np.random.shuffle(affected_users)
affected_users = affected_users[:min(TRAIN_USERS_SAMPLE, len(affected_users))]

train_users, valid_users = train_test_split(affected_users, test_size=0.2, random_state=RANDOM_STATE)
print(f"Пользователей train: {len(train_users)}")
print(f"Пользователей valid: {len(valid_users)}")

# =========================================================
# 9. RETRIEVAL НА OBSERVED TRAIN LOG
# =========================================================
print("\n3. Обучаю retrieval...")

als_obs, bm25_obs, sparse_obs = train_implicit_models(observed_train_log, max_ts)
book_als_obs, sparse_book_obs = train_book_als(observed_train_log, max_ts)

print("4. Строю генераторы...")
i2i_dict_obs = build_i2i_dict(observed_train_log, als_obs, topn=I2I_TOPN_NEIGH)
author_top_obs = build_author_top_items(observed_train_log, recent_days=60, top_per_author=40)
genre_top_obs = build_genre_top_items(observed_train_log, recent_days=60, top_per_genre=50)
book_top_editions_obs = build_book_top_editions(observed_train_log, recent_days=120, top_per_book=3)

recent_obs = observed_train_log[observed_train_log["event_ts"] >= (max_ts - pd.Timedelta(days=30))]
pop_items_obs = recent_obs["edition_id"].value_counts().head(500).index.tolist()

heur_artifacts_obs = build_heuristic_artifacts(observed_train_log, max_ts)
window_artifacts_obs = build_window_artifacts(pseudo_observed)

# =========================================================
# 10. TRAIN / VALID DATASETS
# =========================================================
print("\n5. Собираю train dataset...")
train_df, tr_als, tr_bm25, tr_pop, tr_i2i, tr_author, tr_genre, tr_book, tr_wi2i, tr_wcov, tr_wauth, tr_exp = build_dataset(
    train_users,
    observed_train_log,
    max_ts,
    hidden_pairs,
    als_obs,
    bm25_obs,
    sparse_obs,
    book_als_obs,
    sparse_book_obs,
    i2i_dict_obs,
    author_top_obs,
    genre_top_obs,
    book_top_editions_obs,
    pop_items_obs,
    heur_artifacts_obs,
    window_artifacts_obs
)

print("\n6. Собираю valid dataset...")
valid_df, va_als, va_bm25, va_pop, va_i2i, va_author, va_genre, va_book, va_wi2i, va_wcov, va_wauth, va_exp = build_dataset(
    valid_users,
    observed_train_log,
    max_ts,
    hidden_pairs,
    als_obs,
    bm25_obs,
    sparse_obs,
    book_als_obs,
    sparse_book_obs,
    i2i_dict_obs,
    author_top_obs,
    genre_top_obs,
    book_top_editions_obs,
    pop_items_obs,
    heur_artifacts_obs,
    window_artifacts_obs
)

# =========================================================
# 11. АНАЛИЗ RECALL
# =========================================================
print("\n7. Анализ recall кандидатов...")

truth_valid = hidden_pairs[hidden_pairs["user_id"].isin(valid_users)][["user_id", "edition_id"]].drop_duplicates()

va_als_sorted = va_als.sort_values(["user_id", "score_als"], ascending=[True, False])
va_bm25_sorted = va_bm25.sort_values(["user_id", "score_bm25"], ascending=[True, False])
va_i2i_sorted = va_i2i.sort_values(["user_id", "score_i2i"], ascending=[True, False])
va_author_sorted = va_author.sort_values(["user_id", "score_author"], ascending=[True, False])
va_genre_sorted = va_genre.sort_values(["user_id", "score_genre_gen"], ascending=[True, False])
va_book_sorted = va_book.sort_values(["user_id", "score_book_gen"], ascending=[True, False])

va_wi2i_sorted = va_wi2i.sort_values(["user_id", "score_window_i2i"], ascending=[True, False])
va_wcov_sorted = va_wcov.sort_values(["user_id", "score_window_covisit"], ascending=[True, False])
va_wauth_sorted = va_wauth.sort_values(["user_id", "score_window_author"], ascending=[True, False])
va_exp_sorted = va_exp.sort_values(["user_id", "score_exposure"], ascending=[True, False])

va_union = merge_sources(
    va_als, va_bm25, va_pop, va_i2i, va_author, va_genre, va_book,
    va_wi2i, va_wcov, va_wauth, va_exp
)

score_cols_union = [
    "score_als", "score_bm25", "score_pop", "score_i2i", "score_author",
    "score_genre_gen", "score_book_gen", "score_window_i2i",
    "score_window_covisit", "score_window_author", "score_exposure"
]
for c in score_cols_union:
    if c not in va_union.columns:
        va_union[c] = 0.0

va_union["union_score"] = va_union[score_cols_union].sum(axis=1)
va_union = va_union.sort_values(["user_id", "union_score"], ascending=[True, False])

print(f"ALS Recall@100:            {calc_recall_at_k(va_als_sorted, truth_valid, k=100):.5f}")
print(f"BM25 Recall@100:           {calc_recall_at_k(va_bm25_sorted, truth_valid, k=100):.5f}")
print(f"I2I Recall@100:            {calc_recall_at_k(va_i2i_sorted, truth_valid, k=100):.5f}")
print(f"AUTHOR Recall@100:         {calc_recall_at_k(va_author_sorted, truth_valid, k=100):.5f}")
print(f"GENRE Recall@100:          {calc_recall_at_k(va_genre_sorted, truth_valid, k=100):.5f}")
print(f"BOOK Recall@100:           {calc_recall_at_k(va_book_sorted, truth_valid, k=100):.5f}")
print(f"WINDOW_I2I Recall@100:     {calc_recall_at_k(va_wi2i_sorted, truth_valid, k=100):.5f}")
print(f"WINDOW_COVISIT Recall@100: {calc_recall_at_k(va_wcov_sorted, truth_valid, k=100):.5f}")
print(f"WINDOW_AUTHOR Recall@100:  {calc_recall_at_k(va_wauth_sorted, truth_valid, k=100):.5f}")
print(f"EXPOSURE Recall@100:       {calc_recall_at_k(va_exp_sorted, truth_valid, k=100):.5f}")
print(f"UNION Recall@100:          {calc_recall_at_k(va_union, truth_valid, k=100):.5f}")
print(f"UNION Recall@200:          {calc_recall_at_k(va_union, truth_valid, k=200):.5f}")

# =========================================================
# 12. RANKER
# =========================================================
FEATURES = [
    "score_als",
    "score_bm25",
    "score_pop",
    "score_i2i",
    "score_author",
    "score_genre_gen",
    "score_book_gen",

    "score_window_i2i",
    "score_window_covisit",
    "score_window_author",

    "score_exposure",

    "n_generators_hit",

    "heur_covisit_score",
    "heur_pop_inv_rank",
    "heur_rank_inv",
    "heur_top20",
    "heur_top50",

    "user_act_all",
    "user_act_30d",
    "activity_drop",

    "item_pop_all",
    "item_pop_30d",
    "book_pop_all",
    "book_pop_30d",

    "author_inter_count",
    "same_author_seen_before",
    "genre_score",
    "genre_match_ratio",
    "same_book_seen_before",

    "days_since_user_last_event",
    "days_since_author_last_event",
    "days_since_book_last_event",

    "i2i_max_last3",
    "i2i_mean_last3",
    "i2i_sum_last5",

    "obs_window_same_author_cnt",
    "obs_window_same_book_cnt",
    "obs_window_same_genre_cnt",
    "obs_window_i2i_max",
    "obs_window_covisit_sum",
    "obs_window_item_cnt",

    "window_item_pop",
    "window_pop_rank_inv",
    "exposure_overlap_score",
    "obs_window_activity_ratio",
    "propensity_score",

    "publication_year",
    "age_diff"
]

print("\n8. Обучаю CatBoostRanker...")

train_pool = Pool(
    data=train_df[FEATURES],
    label=train_df["label"],
    group_id=train_df["user_id"],
    weight=train_df["sample_weight"]
)

valid_pool = Pool(
    data=valid_df[FEATURES],
    label=valid_df["label"],
    group_id=valid_df["user_id"],
    weight=valid_df["sample_weight"]
)

ranker = CatBoostRanker(**CB_PARAMS)
ranker.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=200)

print("\n9. Считаю локальный NDCG@20...")
valid_df["pred"] = ranker.predict(valid_df[FEATURES])
valid_df = valid_df.sort_values(
    ["user_id", "pred", "score_exposure", "score_window_i2i", "score_window_covisit", "score_i2i", "score_als"],
    ascending=[True, False, False, False, False, False, False]
).reset_index(drop=True)

local_ndcg = calc_ndcg_per_user(valid_df, score_col="pred", label_col="label", user_col="user_id", k=20)
print(f"Локальный NDCG@20: {local_ndcg:.5f}")

# =========================================================
# 13. ФИНАЛЬНОЕ ДОУЧИВАНИЕ НА ВСЕХ AFFECTED USERS
# =========================================================
print("\n10. Дообучаю финальный ranker на всех affected users...")

full_train_df, *_ = build_dataset(
    affected_users,
    observed_train_log,
    max_ts,
    hidden_pairs,
    als_obs,
    bm25_obs,
    sparse_obs,
    book_als_obs,
    sparse_book_obs,
    i2i_dict_obs,
    author_top_obs,
    genre_top_obs,
    book_top_editions_obs,
    pop_items_obs,
    heur_artifacts_obs,
    window_artifacts_obs
)

full_pool = Pool(
    data=full_train_df[FEATURES],
    label=full_train_df["label"],
    group_id=full_train_df["user_id"],
    weight=full_train_df["sample_weight"]
)

final_ranker = CatBoostRanker(**CB_PARAMS)
final_ranker.fit(full_pool)

# =========================================================
# 14. ФИНАЛЬНЫЙ ИНФЕРЕНС
# =========================================================
print("\n11. Финальный инференс...")

als_full, bm25_full, sparse_full = train_implicit_models(interactions, max_ts)
book_als_full, sparse_book_full = train_book_als(interactions, max_ts)

i2i_dict_full = build_i2i_dict(interactions, als_full, topn=I2I_TOPN_NEIGH)
author_top_full = build_author_top_items(interactions, recent_days=60, top_per_author=40)
genre_top_full = build_genre_top_items(interactions, recent_days=60, top_per_genre=50)
book_top_editions_full = build_book_top_editions(interactions, recent_days=120, top_per_book=3)

recent_full = interactions[interactions["event_ts"] >= (max_ts - pd.Timedelta(days=30))]
pop_items_full = recent_full["edition_id"].value_counts().head(500).index.tolist()

heur_artifacts_full = build_heuristic_artifacts(interactions, max_ts)

# в бою это допустимо: используем реальные наблюдаемые события в окне инцидента
real_window_obs = interactions[
    (interactions["event_ts"] >= INCIDENT_START) &
    (interactions["event_ts"] < INCIDENT_END)
].copy()

window_artifacts_full = build_window_artifacts(real_window_obs)

target_users_list = targets["user_id"].tolist()
empty_labels = pd.DataFrame(columns=["user_id", "edition_id", "label"])

test_df, *_ = build_dataset(
    target_users_list,
    interactions,
    max_ts,
    empty_labels,
    als_full,
    bm25_full,
    sparse_full,
    book_als_full,
    sparse_book_full,
    i2i_dict_full,
    author_top_full,
    genre_top_full,
    book_top_editions_full,
    pop_items_full,
    heur_artifacts_full,
    window_artifacts_full
)

print("12. Считаю предсказание финальной моделью...")
test_df["pred"] = final_ranker.predict(test_df[FEATURES])

# выкидываем уже виденные пары
seen_pairs = interactions[["user_id", "edition_id"]].drop_duplicates()
test_df = test_df.merge(seen_pairs.assign(_seen=1), on=["user_id", "edition_id"], how="left")
test_df = test_df[test_df["_seen"] != 1].drop(columns=["_seen"])

test_df = test_df.sort_values(
    ["user_id", "pred", "score_exposure", "score_window_i2i", "score_window_covisit", "score_i2i", "score_als", "score_author"],
    ascending=[True, False, False, False, False, False, False, False]
).reset_index(drop=True)

print("13. Собираю submission...")
submission_rows = []
grouped = test_df.groupby("user_id")
fallback_items = pop_items_full if len(pop_items_full) > 0 else interactions["edition_id"].value_counts().head(500).index.tolist()

for u in tqdm(target_users_list, desc="submission"):
    preds = grouped.get_group(u)["edition_id"].tolist() if u in grouped.groups else []

    if len(preds) < 20:
        preds.extend([x for x in fallback_items if x not in preds])

    preds = preds[:20]

    for rank, eid in enumerate(preds, 1):
        submission_rows.append({
            "user_id": u,
            "edition_id": eid,
            "rank": rank
        })

submission = pd.DataFrame(submission_rows)

assert submission.groupby("user_id").size().eq(20).all(), "Не у всех user_id ровно 20 строк"
assert not submission.duplicated(["user_id", "edition_id"]).any(), "Есть дубли edition_id внутри user_id"
assert submission["rank"].between(1, 20).all(), "rank вне диапазона 1..20"

submission.to_csv("submission.csv", index=False)
print("\nГотово. submission.csv сохранён.")

# cleanup
del train_df, valid_df, full_train_df, test_df, train_pool, valid_pool, full_pool
gc.collect() #AGI AGI AGI AGI AGI  # 0,1442898766

1. Загружаю данные...
Максимальная дата: 2025-11-30 23:59:35
Реальный инцидент: [2025-10-01 00:00:00 ; 2025-11-01 00:00:00)
Псевдо-инцидент:   [2025-09-01 00:00:00 ; 2025-10-01 00:00:00)

2. Строю правильную локальную симуляцию...
Уникальных позитивных пар в псевдо-окне: 61,211
Скрыто позитивных пар для обучения: 16,100
Наблюдаемый лог для обучения: 427,169 событий
Пользователей train: 5035
Пользователей valid: 1259

3. Обучаю retrieval...


/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.005058765411376953 seconds
  warnings.warn(


4. Строю генераторы...


Строю i2i: 100%|██████████| 123100/123100 [01:13<00:00, 1673.98it/s]


Строю артефакты эвристики для фичей...


...

window_author: 100%|██████████| 7473/7473 [00:00<00:00, 53569.01it/s]


Генерирую exposure кандидаты...


exposure: 100%|██████████| 5035/5035 [00:00<00:00, 6027.10it/s]


Добавляю эвристические фичи...


 20%|██        | 675040/3315510 [02:14<08:08, 5403.83it/s]